## 1. Importações e Configurações de Parâmetros

In [0]:
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("catalogo", "workspace")

catalogo = dbutils.widgets.get("catalogo")

spark.sql(f"USE CATALOG {catalogo}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

TABELA_ORIGEM  = "bronze.tb_clientes"
TABELA_DESTINO = "silver.tb_clientes"

print(f"Catálogo em uso : {catalogo}")
print(f"Origem          : {TABELA_ORIGEM}")
print(f"Destino         : {TABELA_DESTINO}")

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    regexp_replace,
    to_date,
    current_date,
    floor,
    datediff,
    when,
    length,
    concat,
    lit,
    months_between,
    initcap
)

In [0]:
## 2. Leitura da camada Bronze

In [0]:
# Todos os campos chegam como STRING da Bronze (inferSchema=false).
# A tipagem correta é feita neste notebook de forma controlada e rastreável.
inicio_total = time.time()
execucao_id  = datetime.now().strftime("%Y%m%d_%H%M%S")

df_bronze = spark.table(TABELA_ORIGEM)

total_bronze = df_bronze.count()
print(f"Registros lidos da Bronze: {total_bronze:,}")
print(f"Colunas: {df_bronze.columns}")

In [0]:
df_silver = df_bronze

In [0]:
df_silver = df_silver \
    .withColumn("data_nascimento", to_date(col("data_nascimento"), "yyyy-MM-dd")) \
    .withColumn("data_cadastro", to_date(col("data_cadastro"), "yyyy-MM-dd"))

# Mostrar como ficou a estrutura
df_silver.printSchema()

In [0]:
print("Contagem de valores NULOS por coluna:")

# Cria uma lista de expressões para contar os nulos
expressoes_nulos = [
    F.sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_silver.columns
]

# Executa e mostra o resultado
df_silver.select(*expressoes_nulos).display()

In [0]:
# Ver o que tem no dado bruto, antes de qualquer limpeza
display(df_bronze.filter(col("device_ids").isNull()))

In [0]:
display(df_silver.filter(col("device_ids").isNull()))

In [0]:
df_silver = df_silver.fillna({"device_ids": "Não Informado"})

In [0]:
# 1. Busca IDs duplicados
ids_duplicados = df_silver.groupBy("id_cliente").count().filter("count > 1")

# 2. Faz o Join usando alias para evitar conflitos
df_apenas_duplicados = df_silver.alias("origem").join(
    ids_duplicados.alias("dups"), 
    on="id_cliente", 
    how="inner"
)

# 3. Exibe o resultado
display(df_apenas_duplicados.orderBy("id_cliente"))

In [0]:
# Guardando a quantidade original para comparar
total_antes = df_silver.count()

# Apagando as duplicatas (mantém apenas a primeira ocorrência de cada id_cliente)
df_silver = df_silver.dropDuplicates(["id_cliente"])

# Contando após a limpeza
total_depois = df_silver.count()

# Exibindo o resultado da faxina
print(f"Linhas antes da limpeza: {total_antes}")
print(f"Linhas depois da limpeza: {total_depois}")
print(f"Total de duplicadas removidas: {total_antes - total_depois}")

In [0]:
display(df_silver)

In [0]:
df_silver.select(
    F.max("data_cadastro")
).show()

In [0]:
df_silver.select(
    F.max("data_nascimento")
).show()

In [0]:
df_silver.select(
    F.min("data_nascimento")
).show()

In [0]:
# O regex [^0-9\+r\.] apaga qualquer coisa que NÃO seja (^) número (0-9), o mais (\+), a letra r ou o ponto (\.)
df_silver = df_silver.withColumn(
    "telefone", 
    regexp_replace(col("telefone"), r"[^0-9\+r]", "")
)

display(df_silver.select("telefone"))

In [0]:
df_silver = df_silver.withColumn(
    "telefone",
    when(
        col("telefone").startswith("+55"),
        col("telefone")
    ).otherwise(
        concat(lit("+55"), col("telefone"))
    )
)

display(df_silver.select("telefone"))

In [0]:
# Remove tudo a partir do "r" (ramal)
df_silver = df_silver.withColumn(
    "telefone",
    regexp_replace(col("telefone"), r"r\d+", "")
)

display(df_silver.select("telefone"))

In [0]:
# Adicionando a coluna 'idade'
df_silver = df_silver.withColumn(
    "idade", 
    floor(months_between(current_date(), col("data_nascimento")) / 12).cast("integer")
)

# Visualizando o resultado para conferir
display(df_silver.select("nome", "data_nascimento", "idade"))

In [0]:
# trim: remove espaços no início e no fim
# initcap: deixa a primeira letra de cada palavra maiúscula (Ex: " joão silva " -> "João Silva")
df_silver = df_silver.withColumn("nome", initcap(trim(col("nome"))))
df_silver = df_silver.withColumn("sobrenome", initcap(trim(col("sobrenome"))))

In [0]:
from pyspark.sql.functions import lower

# lower: deixa todas as letras minúsculas (Ex: " Maria@GMAIL.com " -> "maria@gmail.com")
df_silver = df_silver.withColumn("email", lower(trim(col("email"))))

In [0]:
display(df_silver)

In [0]:
# Verificado o máx e minimo das avaliações
df_silver.select(
    F.max("idade")
).show()

df_silver.select(
    F.min("idade")
).show()

In [0]:
# Traz apenas os valores únicos da coluna origem
df_silver.select("origem").distinct().display()

In [0]:
df_silver = df_silver.withColumn(
    "origem",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                lower(trim(col("origem"))),
                "[ç]", "c"
            ),
            "[ãáàâ]", "a"
        ),
        "[íìî]", "i"
    )
)

In [0]:
# Traz apenas os valores únicos da coluna origem
df_silver.select("origem").distinct().display()

In [0]:
# Traz apenas os valores únicos da coluna origem
df_silver.select("pais").distinct().display()

In [0]:
display(df_silver)

In [0]:
#Salvando na Camada Silver
TABELA_DESTINO_CLIENTES = "silver.tb_clientes"
df_silver_clientes = df_silver 
df_silver_clientes.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_DESTINO_CLIENTES)

print(f" Tabela de clientes salva com sucesso em: {TABELA_DESTINO_CLIENTES}")
display(df_silver_clientes.limit(5))